## Dubai Land Department — Residential Sales EDA & Cleaning Pipeline

This notebook takes the raw DLD dataset from CSV to a clean, analysis-ready parquet snapshot.

The process follows a deliberate sequence: first understand what we have, then remove what we do not need, then reshape what remains into a form that is consistent, interpretable, and fast to query.

Each step is documented with the reasoning behind it. By the end of this notebook, 1.6M raw rows become ~566K clean residential sales records — every row a real transaction, every column meaningful.

## How to Use This Project

This project is structured as three sequential notebooks. Each one builds on the output of the previous.

| Notebook | Purpose | Input | Output |
|---|---|---|---|
| `01_eda.ipynb` | Data cleaning pipeline | `dld-raw.csv` (1.66M rows) | `dld-clean.parquet` (566K rows) |
| `02_analysis.ipynb` | Statistical analysis and metrics | `dld-clean.parquet` | Insights and charts |
| `03_insights.ipynb` | Visual storytelling | `dld-clean.parquet` | Publication-ready charts with written analysis |

**To reproduce the project from scratch:**
1. Run `01_eda.ipynb` top to bottom. This produces the clean parquet snapshot. Update `DATA_PATH` and `SNAPSHOT_PATH` to match your local file paths.
2. Run `02_analysis.ipynb` top to bottom. Update `DATA_PATH` to point to your parquet file.
3. Run `03_insights.ipynb` top to bottom. Same `DATA_PATH` update applies.

**Stack:** Python, Polars, Plotly Express, Plotly Graph Objects.

**Dataset:** Official Dubai Land Department (DLD) residential transaction registry. Publicly available. The raw file is not included in this repository due to size.

In [1]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
pl.Config.set_tbl_rows(100)

polars.config.Config

## Step 1: Load the Raw Data

We load the CSV using Polars in lazy-eager mode with `infer_schema_length=10000`, which tells Polars to scan the first 10,000 rows before deciding column types. The default of 100 rows is often too small for a dataset this large and causes type mismatches later when rare values appear further down.

`null_values=[""]` ensures that empty strings are treated as proper nulls rather than empty text, which matters for the null audit in the next step.

In [2]:
DATA_PATH = r"E:\Databard\DLD\dld-raw.csv"

df = pl.read_csv(
    DATA_PATH,
    infer_schema_length=10000,
    null_values=["", "null", "NULL", "N/A"],
    try_parse_dates=False,   # we'll parse manually — format is DD-MM-YYYY
)

print(f"Rows: {df.height:,}")
print(f"Columns: {df.width}")

Rows: 1,665,112
Columns: 46


## Step 2: Schema Inspection

Before touching the data, we print the full schema to understand what Polars inferred for each column. This tells us which columns arrived as strings that should be dates or numbers, which are already typed correctly, and where we will need to cast or transform later.

Think of this as reading the table of contents before starting a book.

In [3]:
print(df.schema)

Schema({'transaction_id': String, 'procedure_id': Int64, 'trans_group_id': Int64, 'trans_group_ar': String, 'trans_group_en': String, 'procedure_name_ar': String, 'procedure_name_en': String, 'instance_date': String, 'property_type_id': Int64, 'property_type_ar': String, 'property_type_en': String, 'property_sub_type_id': Int64, 'property_sub_type_ar': String, 'property_sub_type_en': String, 'property_usage_ar': String, 'property_usage_en': String, 'reg_type_id': Int64, 'reg_type_ar': String, 'reg_type_en': String, 'area_id': Int64, 'area_name_ar': String, 'area_name_en': String, 'building_name_ar': String, 'building_name_en': String, 'project_number': Int64, 'project_name_ar': String, 'project_name_en': String, 'master_project_en': String, 'master_project_ar': String, 'nearest_landmark_ar': String, 'nearest_landmark_en': String, 'nearest_metro_ar': String, 'nearest_metro_en': String, 'nearest_mall_ar': String, 'nearest_mall_en': String, 'rooms_ar': String, 'rooms_en': String, 'has_par

## Step 3: Null Audit

We count nulls across every column and express them as a percentage of total rows. This is a non-negotiable first step before any cleaning decision.

A column with 90% nulls is almost certainly useless and should be dropped. A column with 5% nulls needs a decision: impute, filter, or carry the nulls forward. A column with 0% nulls can be trusted completely. Without this audit, we are making cleaning decisions blind.

In [4]:
null_counts = (
    df.select([
        pl.col(c).is_null().sum().alias(c)
        for c in df.columns
    ])
    .unpivot(variable_name="column", value_name="nulls")
    .with_columns([
        (pl.col("nulls") / df.height * 100).round(2).alias("null_pct")
    ])
    .sort("nulls", descending=True)
)

print(null_counts)

shape: (46, 3)
┌──────────────────────┬─────────┬──────────┐
│ column               ┆ nulls   ┆ null_pct │
│ ---                  ┆ ---     ┆ ---      │
│ str                  ┆ u32     ┆ f64      │
╞══════════════════════╪═════════╪══════════╡
│ rent_value           ┆ 1628561 ┆ 97.8     │
│ meter_rent_price     ┆ 1628561 ┆ 97.8     │
│ nearest_mall_ar      ┆ 512846  ┆ 30.8     │
│ nearest_mall_en      ┆ 512846  ┆ 30.8     │
│ nearest_metro_ar     ┆ 503033  ┆ 30.21    │
│ nearest_metro_en     ┆ 503033  ┆ 30.21    │
│ building_name_ar     ┆ 480649  ┆ 28.87    │
│ building_name_en     ┆ 480193  ┆ 28.84    │
│ project_number       ┆ 456662  ┆ 27.43    │
│ project_name_ar      ┆ 456662  ┆ 27.43    │
│ project_name_en      ┆ 456662  ┆ 27.43    │
│ rooms_ar             ┆ 358408  ┆ 21.52    │
│ rooms_en             ┆ 358408  ┆ 21.52    │
│ property_sub_type_id ┆ 334703  ┆ 20.1     │
│ property_sub_type_ar ┆ 334703  ┆ 20.1     │
│ property_sub_type_en ┆ 334703  ┆ 20.1     │
│ nearest_landmark_

## Step 4: Sample Rows

We pull 10 random rows with a fixed seed to get a feel for what the data actually looks like in practice. Schema tells you types; sample rows tell you values.

This is where you spot inconsistencies that a schema cannot: inconsistent casing, mixed date formats, placeholder strings like "N/A" that were not caught as nulls, and columns whose content differs from what the name implies.

In [5]:
df.sample(10, seed=42)

transaction_id,procedure_id,trans_group_id,trans_group_ar,trans_group_en,procedure_name_ar,procedure_name_en,instance_date,property_type_id,property_type_ar,property_type_en,property_sub_type_id,property_sub_type_ar,property_sub_type_en,property_usage_ar,property_usage_en,reg_type_id,reg_type_ar,reg_type_en,area_id,area_name_ar,area_name_en,building_name_ar,building_name_en,project_number,project_name_ar,project_name_en,master_project_en,master_project_ar,nearest_landmark_ar,nearest_landmark_en,nearest_metro_ar,nearest_metro_en,nearest_mall_ar,nearest_mall_en,rooms_ar,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price,rent_value,meter_rent_price,no_of_parties_role_1,no_of_parties_role_2,no_of_parties_role_3
str,i64,i64,str,str,str,str,str,i64,str,str,i64,str,str,str,str,i64,str,str,i64,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,str,str,i64,f64,f64,f64,f64,f64,i64,i64,i64
"""1-102-2007-23907""",102,1,"""مبايعات""","""Sales""","""بيع - تسجيل مبدئى""","""Sell - Pre registration""","""28-01-2009""",3,"""وحدة""","""Unit""",60,"""شقه سكنيه""","""Flat""","""سكني""","""Residential""",0,"""على الخارطة""","""Off-Plan Properties""",441,"""البرشاء جنوب الرابعة""","""Al Barsha South Fourth""","""لاريفيرا استيتس - بي""","""LARIVIERA ESTATE-B""",null,null,null,"""Jumeirah Village Circle""","""قرية جميرا الدائرية""","""أكاديمية المدينة الرياضية للسب…","""Sports City Swimming Academy""","""مدينة دبي للإنترنت""","""Dubai Internet City""","""مارينا مول""","""Marina Mall""","""غرفة""","""1 B/R""",1,83.03,630000.0,7587.62,null,null,1,1,0
"""1-102-2021-2940""",102,1,"""مبايعات""","""Sales""","""بيع - تسجيل مبدئى""","""Sell - Pre registration""","""17-03-2021""",3,"""وحدة""","""Unit""",60,"""شقه سكنيه""","""Flat""","""سكني""","""Residential""",0,"""على الخارطة""","""Off-Plan Properties""",452,"""الحبيه الثانية""","""Al Hebiah Second""","""سامانا جولف افنيو""","""Samana Golf Avenue""",2202,"""سامانا جولف افنيو""","""SAMANA GOLF AVENUE""","""Dubai Studio City""","""مدينة دبي للستديو""","""موتور سيتي""","""Motor City""",null,null,null,null,"""استوديو""","""Studio""",1,37.97,438500.0,11548.59,null,null,1,1,0
"""2-110-2012-156""",110,2,"""رهون""","""Mortgages""","""تسجيل إيجارة تنتهى بالتملك""","""Lease to Own Registration""","""07-03-2012""",3,"""وحدة""","""Unit""",42,"""مكتب""","""Office""","""تجاري""","""Commercial""",1,"""العقارات القائمة""","""Existing Properties""",350,"""الثنيه الخامسة""","""Al Thanyah Fifth""","""جميرا باي تاور اكس 2""","""JUMAIRAH BAY TOWER-X2""",null,null,null,"""Jumeirah Lakes Towers""",""" ابراج بحيرات الجميرا""","""برج العرب""","""Burj Al Arab""","""عقارات داماك""","""Damac Properties""","""مارينا مول""","""Marina Mall""","""مكتب""","""Office""",1,55.86,325242.0,5822.45,325242.0,5822.45,2,2,2
"""2-13-2008-2295""",13,2,"""رهون""","""Mortgages""","""تسجيل رهن""","""Mortgage Registration""","""23-09-2008""",4,"""فيلا""","""Villa""",null,null,null,"""سكني""","""Residential""",1,"""العقارات القائمة""","""Existing Properties""",264,"""ند الحمر""","""Nad Al Hamar""",null,null,null,null,null,null,null,"""مطار دبي الدولي""","""Dubai International Airport""","""محطة مترو الراشدية""","""Rashidiya Metro Station""","""سيتي سنتر مردف""","""City Centre Mirdif""",null,null,0,1896.34,500000.0,263.67,null,null,1,1,0
"""1-11-2020-33""",11,1,"""مبايعات""","""Sales""","""بيع""","""Sell""","""02-01-2020""",3,"""وحدة""","""Unit""",60,"""شقه سكنيه""","""Flat""","""سكني""","""Residential""",1,"""العقارات القائمة""","""Existing Properties""",343,"""ورسان الاولى""","""Al Warsan First""","""اف - 15""","""F-15""",null,null,null,"""International City Phase 1""","""المدينة العالمية - المرحلة الا…",null,null,"""محطة مترو الراشدية""","""Rashidiya Metro Station""","""سيتي سنتر مردف""","""City Centre Mirdif""","""غرفة""","""1 B/R""",0,62.24,300000.0,4820.05,null,null,1,1,0
"""1-102-2025-122453""",102,1,"""مبايعات""","""Sales""","""بيع - تسجيل مبدئى""","""Sell - Pre registration""","""28-11-2025""",3,"""وحدة""","""Unit""",60,"""

## Step 5: Drop Irrelevant Columns

The raw dataset has 46 columns. Many are redundant, administrative, or carry no analytical value for this project.

We drop in three categories:
- **Arabic duplicates** — every English column has an Arabic mirror. We are doing an English-language analysis, so these add nothing but width.
- **Zero-variance or near-zero columns** — columns where the null audit showed 90%+ nulls, or where every row has the same value, contribute nothing to any analysis.
- **Administrative metadata** — internal DLD identifiers, transaction IDs, and registration codes that have no interpretive meaning for a market analysis.

Fewer columns means faster operations, a cleaner schema, and less cognitive overhead when inspecting the data later.

In [6]:
COLS_TO_DROP = [
    # Arabic duplicates
    "trans_group_ar", "procedure_name_ar", "property_type_ar",
    "property_sub_type_ar", "property_usage_ar", "reg_type_ar",
    "area_name_ar", "building_name_ar", "project_name_ar",
    "master_project_ar", "nearest_landmark_ar", "nearest_metro_ar",
    "nearest_mall_ar", "rooms_ar",
    # Redundant numeric IDs
    "procedure_id", "trans_group_id", "property_type_id",
    "property_sub_type_id", "reg_type_id", "area_id",
    # Low-value party counts
    "no_of_parties_role_1", "no_of_parties_role_2", "no_of_parties_role_3", "transaction_id",
    # Mostly null
    "rent_value", "meter_rent_price"
]

df = df.drop(COLS_TO_DROP)

print(f"Columns remaining: {df.width}")
print(df.columns)

Columns remaining: 20
['trans_group_en', 'procedure_name_en', 'instance_date', 'property_type_en', 'property_sub_type_en', 'property_usage_en', 'reg_type_en', 'area_name_en', 'building_name_en', 'project_number', 'project_name_en', 'master_project_en', 'nearest_landmark_en', 'nearest_metro_en', 'nearest_mall_en', 'rooms_en', 'has_parking', 'procedure_area', 'actual_worth', 'meter_sale_price']


In [7]:
df.sample(10, seed=42)

trans_group_en,procedure_name_en,instance_date,property_type_en,property_sub_type_en,property_usage_en,reg_type_en,area_name_en,building_name_en,project_number,project_name_en,master_project_en,nearest_landmark_en,nearest_metro_en,nearest_mall_en,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price
str,str,str,str,str,str,str,str,str,i64,str,str,str,str,str,str,i64,f64,f64,f64
"""Sales""","""Sell - Pre registration""","""28-01-2009""","""Unit""","""Flat""","""Residential""","""Off-Plan Properties""","""Al Barsha South Fourth""","""LARIVIERA ESTATE-B""",null,null,"""Jumeirah Village Circle""","""Sports City Swimming Academy""","""Dubai Internet City""","""Marina Mall""","""1 B/R""",1,83.03,630000.0,7587.62
"""Sales""","""Sell - Pre registration""","""17-03-2021""","""Unit""","""Flat""","""Residential""","""Off-Plan Properties""","""Al Hebiah Second""","""Samana Golf Avenue""",2202,"""SAMANA GOLF AVENUE""","""Dubai Studio City""","""Motor City""",null,null,"""Studio""",1,37.97,438500.0,11548.59
"""Mortgages""","""Lease to Own Registration""","""07-03-2012""","""Unit""","""Office""","""Commercial""","""Existing Properties""","""Al Thanyah Fifth""","""JUMAIRAH BAY TOWER-X2""",null,null,"""Jumeirah Lakes Towers""","""Burj Al Arab""","""Damac Properties""","""Marina Mall""","""Office""",1,55.86,325242.0,5822.45
"""Mortgages""","""Mortgage Registration""","""23-09-2008""","""Villa""",null,"""Residential""","""Existing Properties""","""Nad Al Hamar""",null,null,null,null,"""Dubai International Airport""","""Rashidiya Metro Station""","""City Centre Mirdif""",null,0,1896.34,500000.0,263.67
"""Sales""","""Sell""","""02-01-2020""","""Unit""","""Flat""","""Residential""","""Existing Properties""","""Al Warsan First""","""F-15""",null,null,"""International City Phase 1""",null,"""Rashidiya Metro Station""","""City Centre Mirdif""","""1 B/R""",0,62.24,300000.0,4820.05
"""Sales""","""Sell - Pre registration""","""28-11-2025""","""Unit""","""Flat""","""Residential""","""Off-Plan Properties""","""Madinat Dubai Almelaheyah""","""Franck Muller Yachting""",4017,"""Franck Muller Yachting""","""Dubai Maritime City""","""Burj Khalifa""","""Al Ghubaiba Metro Station""","""Dubai Mall""","""Studio""",1,39.86,1.266e6,31761.16
"""Sales""","""Sell""","""03-02-2022""","""Villa""","""Villa""","""Residential""","""Existing Properties""","""Al Thanayah Fourth""",null,1050,"""Emirates Living - Springs 11""","""Springs - 1""","""Sports City Swimming Academy""","""Damac Properties""","""Marina Mall""","""2 B/R""",0,171.36,1.75e6,10212.42
"""Sales""","""Sell - Pre registration""","""29-06-2020""","""Villa""","""Villa""","""Residential""","""Off-Plan Properties""","""Wadi Al Safa 5""",null,1817,"""Villanova Amaranta """,null,"""IMG World Adventures""",null,null,"""4 B/R""",0,231.87,1.53e6,6598.53
"""Sales""","""Sell - Pre registration""","""22-12-2025""","""Villa""","""Villa""","""Residential""","""Off-Plan Properties""","""Dubai Investment Park Second""",null,4185,"""Grand Polo - Equiterra """,null,null,null,null,"""4 B/R""",0,250.37,4.618888e6,18448.25


## Step 6: Filter to Post-2020 Data

The raw dataset goes back decades. We keep only transactions from 1 January 2021 onwards.

The reason is relevance. Dubai's property market before COVID-19 operated under fundamentally different conditions: lower population, different visa rules, less international capital, and a market that had just gone through a multi-year correction. Including pre-2021 data would distort every trend, average, and comparison we build later. The story we want to tell is the post-pandemic cycle, so we start where that cycle starts.

This also converts `instance_date` from a raw string to a proper Polars `Date` type, which unlocks date arithmetic throughout the rest of the notebook.

In [8]:
df = (
    df
    .with_columns(
        pl.col("instance_date")
          .str.to_date(format="%d-%m-%Y", strict=False)
          .alias("instance_date")
    )
    .filter(pl.col("instance_date") >= pl.date(2021, 1, 1))
)

print(f"Rows after date filter: {df.height:,}")
print(f"Date range: {df['instance_date'].min()} → {df['instance_date'].max()}")

Rows after date filter: 894,123
Date range: 2021-01-01 → 2026-02-17


## Step 7: Extract Year and Month

We derive `year` and `month` as separate integer columns from the date.

Storing these explicitly avoids repeating `dt.year()` and `dt.month()` extractions in every downstream group-by. It also makes the data immediately readable when sampling rows: you see `2023 / 4` at a glance rather than having to parse a date string.

In [9]:
df = df.with_columns([
    pl.col("instance_date").dt.year().alias("year"),
    pl.col("instance_date").dt.month().alias("month"),
])

print(df.select(["instance_date", "year", "month"]).head(5))
df.sample(10, seed=42)

shape: (5, 3)
┌───────────────┬──────┬───────┐
│ instance_date ┆ year ┆ month │
│ ---           ┆ ---  ┆ ---   │
│ date          ┆ i32  ┆ i8    │
╞═══════════════╪══════╪═══════╡
│ 2026-01-05    ┆ 2026 ┆ 1     │
│ 2025-09-09    ┆ 2025 ┆ 9     │
│ 2025-09-25    ┆ 2025 ┆ 9     │
│ 2024-10-28    ┆ 2024 ┆ 10    │
│ 2024-07-30    ┆ 2024 ┆ 7     │
└───────────────┴──────┴───────┘


trans_group_en,procedure_name_en,instance_date,property_type_en,property_sub_type_en,property_usage_en,reg_type_en,area_name_en,building_name_en,project_number,project_name_en,master_project_en,nearest_landmark_en,nearest_metro_en,nearest_mall_en,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price,year,month
str,str,date,str,str,str,str,str,str,i64,str,str,str,str,str,str,i64,f64,f64,f64,i32,i8
"""Sales""","""Sell - Pre registration""",2023-04-04,"""Unit""","""Hotel Rooms""","""Hospitality ""","""Off-Plan Properties""","""Marsa Dubai""","""FIVE LUXE""",2309,"""FIVE LUXE""",null,"""Burj Al Arab""","""Jumeirah Beach Resdency""","""Marina Mall""","""1 B/R""",1,115.64,5.714298e6,49414.55,2023,4
"""Mortgages""","""Mortgage Registration""",2024-08-02,"""Villa""","""Villa""","""Residential""","""Existing Properties""","""Al Yelayiss 2""",null,1637,"""TOWN SQUARE ZAHRA""","""TOWN SQUARE""","""Dubai Cycling Course""",null,null,"""3 B/R""",0,202.5,1.88e6,9283.95,2024,8
"""Sales""","""Sell""",2022-05-13,"""Unit""","""Flat""","""Residential""","""Existing Properties""","""Al Barshaa South Third""","""MIRACLZ TOWER""",1819,"""MIRACLZ TOWER""","""Arjan""","""Motor City""","""Sharaf Dg Metro Station""","""Mall of the Emirates""","""1 B/R""",1,70.48,605000.0,8584.0,2022,5
"""Sales""","""Sell - Pre registration""",2025-03-05,"""Unit""","""Flat""","""Residential""","""Off-Plan Properties""","""Al Hebiah Fourth""","""The Community-Sports Arena""",3362,"""The Community - Sports Arena""","""Dubai Sports City""","""Sports City Swimming Academy""","""Dubai Internet City""","""Marina Mall""","""1 B/R""",1,88.13,1.188888e6,13490.16,2025,3
"""Sales""","""Sell - Pre registration""",2022-09-13,"""Villa""","""Villa""","""Residential""","""Off-Plan Properties""","""Wadi Al Safa 5""",null,2348,"""La Violeta 1""",null,"""IMG World Adventures""",null,null,"""4 B/R""",0,253.13,1.885e6,7446.77,2022,9
"""Sales""","""Delayed Sell""",2023-06-21,"""Land""",null,"""Residential""","""Existing Properties""","""Al Hebiah Fifth""",null,2486,"""DAMAC LAGOONS - IBIZA""",null,null,null,null,null,0,144.17,2.269e6,15738.36,2023,6
"""Sales""","""Sell - Pre registration""",2023-03-15,"""Unit""","""Flat""","""Residential""","""Off-Plan Properties""","""Al Barsha South Fourth""","""Binghatti Nova""",2494,"""Binghatti Nova""","""Jumeirah Village Circle""","""Sports City Swimming Academy""","""Dubai Internet City""","""Mall of the Emirates""","""1 B/R""",1,59.0,630000.0,10677.97,2023,3
"""Sales""","""Sell - Pre registration""",2023-09-26,"""Unit""","""Flat""","""Residential""","""Off-Plan Properties""","""Hadaeq Sheikh Mohammed Bin Ras…","""Greenside Residence - Tower A""",2705,"""Greenside Residence""","""Dubai Hills Estate""","""IMG World Adventures""","""Noor Bank Metro Station""","""Mall of the Emirates""","""1 B/R""",1,67.33,1.486888e6,22083.59,2023,9
"""Sales""","""Sell - Pre registration""",2025-02-06,"""Villa""","""Villa""","""Residential""","""Off-Plan Properties""","""Wadi Al Safa 5""",null,2874,"""Haven By Aldar 2""",null,null,null,null,"""4 B/R""",0,319.98,4.1e6,12813.3,2025,2


## Step 8: Filter to Sales Transactions Only

The DLD dataset includes all registered real estate transactions: sales, mortgages, gifts, inheritances, court orders, and more. We only want sales.

Keeping non-sale transaction types would contaminate every price metric. A mortgage registration does not tell us what a property sold for. A gift transfer does not reflect market value at all. We filter to `trans_group_en == "Sales"` and drop the column immediately after, since every remaining row now has the same value.

In [10]:
df = df.filter(pl.col("trans_group_en") == "Sales").drop("trans_group_en")

print(f"Rows after sales filter: {df.height:,}")
print(df["procedure_name_en"].value_counts().sort("count", descending=True))

Rows after sales filter: 708,785
shape: (18, 2)
┌─────────────────────────────────┬────────┐
│ procedure_name_en               ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ Sell - Pre registration         ┆ 391964 │
│ Sell                            ┆ 208052 │
│ Delayed Sell                    ┆ 92424  │
│ Sell Development                ┆ 6546   │
│ Lease to Own Registration       ┆ 3594   │
│ Development Registration        ┆ 2187   │
│ Sale On Payment Plan            ┆ 1829   │
│ Development Registration Pre-R… ┆ 1300   │
│ Delayed Development             ┆ 281    │
│ Delayed Sell Lease to Own Regi… ┆ 231    │
│ Delayed Sell Development        ┆ 125    │
│ Sell Development - Pre Registr… ┆ 101    │
│ Adding Land By Sell             ┆ 44     │
│ Lease to Own Registration Pre-… ┆ 38     │
│ Delayed Lease to Own Registrat… ┆ 26     │
│ Lease to Own on Development Re… ┆ 26     │
│ Lease

## Step 9: Filter to Meaningful Sale Procedure Types

Within the "Sales" transaction group there are several procedure sub-types. We keep only three: `Sell`, `Sell - Pre registration`, and `Delayed Sell`.

The rest are edge cases: bulk transfers, court-ordered sales, or administrative re-registrations that do not represent a buyer and seller agreeing on a market price. Including them would introduce noise into price metrics without adding any meaningful volume.

In [11]:
KEEP_PROCEDURES = ["Sell - Pre registration", "Sell", "Delayed Sell"]

df = df.filter(pl.col("procedure_name_en").is_in(KEEP_PROCEDURES))

print(f"Rows after procedure filter: {df.height:,}")
print(df["procedure_name_en"].value_counts().sort("count", descending=True))

Rows after procedure filter: 692,440
shape: (3, 2)
┌─────────────────────────┬────────┐
│ procedure_name_en       ┆ count  │
│ ---                     ┆ ---    │
│ str                     ┆ u32    │
╞═════════════════════════╪════════╡
│ Sell - Pre registration ┆ 391964 │
│ Sell                    ┆ 208052 │
│ Delayed Sell            ┆ 92424  │
└─────────────────────────┴────────┘


## Step 10: Derive Off-Plan vs Ready from Procedure Type

The procedure name encodes sale type directly: `Sell - Pre registration` and `Delayed Sell` both represent off-plan purchases (buyer commits before the unit is built), while `Sell` is a completed unit changing hands on the secondary or primary ready market.

Rather than carry the verbose procedure string forward, we encode this as a clean binary `sale_type` column with values `"Off-Plan"` and `"Ready"`. This collapses three procedure types into one interpretable dimension and removes a column whose information is now fully captured elsewhere.

In [12]:
df = df.with_columns(
    pl.when(pl.col("procedure_name_en") == "Sell")
      .then(pl.lit("Ready"))
      .otherwise(pl.lit("Off-Plan"))
      .alias("sale_type")
)

print(df["sale_type"].value_counts().sort("count", descending=True))

shape: (2, 2)
┌───────────┬────────┐
│ sale_type ┆ count  │
│ ---       ┆ ---    │
│ str       ┆ u32    │
╞═══════════╪════════╡
│ Off-Plan  ┆ 484388 │
│ Ready     ┆ 208052 │
└───────────┴────────┘


In [13]:
df = df.drop("procedure_name_en", "reg_type_en")

print(df.columns)
df.sample(10, seed=42)

['instance_date', 'property_type_en', 'property_sub_type_en', 'property_usage_en', 'area_name_en', 'building_name_en', 'project_number', 'project_name_en', 'master_project_en', 'nearest_landmark_en', 'nearest_metro_en', 'nearest_mall_en', 'rooms_en', 'has_parking', 'procedure_area', 'actual_worth', 'meter_sale_price', 'year', 'month', 'sale_type']


instance_date,property_type_en,property_sub_type_en,property_usage_en,area_name_en,building_name_en,project_number,project_name_en,master_project_en,nearest_landmark_en,nearest_metro_en,nearest_mall_en,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price,year,month,sale_type
date,str,str,str,str,str,i64,str,str,str,str,str,str,i64,f64,f64,f64,i32,i8,str
2026-01-15,"""Villa""",null,"""Commercial""","""Al Hebiah Fourth""",null,1367,"""PRIME VILLAS""","""Dubai Sports City""","""Sports City Swimming Academy""","""Nakheel Metro Station""","""Marina Mall""",null,0,365.97,5.5e6,15028.55,2026,1,"""Ready"""
2025-09-25,"""Unit""","""Flat""","""Residential""","""Wadi Al Safa 5""","""Skycourts Tower D""",794,"""SKY COURTS""","""Dubai Land Residence Complex""","""IMG World Adventures""",null,null,"""Studio""",1,50.77,425000.0,8371.09,2025,9,"""Ready"""
2022-11-10,"""Unit""","""Flat""","""Residential""","""Jabal Ali First""","""THE NOOK 1-1""",2086,"""The Nook""","""Wasl Gate""","""Expo 2020 Site""","""ENERGY Metro Station""","""Ibn-e-Battuta Mall""","""2 B/R""",1,69.21,921777.0,13318.55,2022,11,"""Off-Plan"""
2025-08-25,"""Unit""","""Flat""","""Residential""","""Al Thanyah Fifth""","""LAKE VIEW""",437,"""LAKE VIEW""","""Jumeirah Lakes Towers""","""Sports City Swimming Academy""","""Jumeirah Lakes Towers""","""Marina Mall""","""Studio""",1,39.02,750000.0,19220.91,2025,8,"""Ready"""
2025-03-05,"""Unit""","""Flat""","""Residential""","""Jabal Ali First""","""South Garden A""",3138,"""South Garden A""","""Wasl Gate""","""Expo 2020 Site""","""ENERGY Metro Station""","""Ibn-e-Battuta Mall""","""1 B/R""",1,79.71,939000.0,11780.2,2025,3,"""Off-Plan"""
2024-10-07,"""Unit""","""Flat""","""Residential""","""Business Bay""","""Rove Home Marasi Drive""",2966,"""Rove Home Marasi Drive""","""Business Bay""","""Downtown Dubai""","""Business Bay Metro Station""","""Dubai Mall""","""2 B/R""",1,97.07,2.420888e6,24939.61,2024,10,"""Off-Plan"""
2023-06-12,"""Unit""","""Flat""","""Residential""","""Al Barsha South Fourth""","""KNIGHTSBRIDGE COURT""",637,"""KNIGHT BRIDGE COURT & KENSNGTO…","""Jumeirah Village Circle""","""Sports City Swimming Academy""","""Nakheel Metro Station""","""Marina Mall""","""Studio""",1,33.59,280000.0,8335.81,2023,6,"""Ready"""
2024-06-21,"""Unit""","""Flat""","""Residential""","""Wadi Al Safa 5""","""COVE LIVING RESIDENCE BY IMTIA…",2973,"""Cove Living Residence By Imtia…","""Dubai Land Residence Complex""","""IMG World Adventures""",null,null,"""Studio""",1,59.71,747491.0,12518.7,2024,6,"""Off-Plan"""
2024-04-22,"""Unit""","""Flat""","""Residential""","""Al Khairan First""","""Oria""",2941,"""Oria""","""Dubai Creek Harbour""",null,null,null,"""2 B/R""",1,120.75,2.795888e6,23154.35,2024,4,"""Off-Plan"""


## Step 11: Inspect Property Type and Sub-Type

Before we can build a clean property category column, we need to understand what values actually exist in `property_type_en` and `property_sub_type_en` and how they relate to each other.

We print value counts for both to see which combinations are common, which are rare, and which are likely noise or mis-classifications. This is the basis for the mapping decision in the next step.

In [14]:
print("property_type_en:")
print(df["property_type_en"].value_counts().sort("count", descending=True))

print("\nproperty_sub_type_en:")
print(df["property_sub_type_en"].value_counts().sort("count", descending=True))

property_type_en:
shape: (4, 2)
┌──────────────────┬────────┐
│ property_type_en ┆ count  │
│ ---              ┆ ---    │
│ str              ┆ u32    │
╞══════════════════╪════════╡
│ Unit             ┆ 541623 │
│ Villa            ┆ 79907  │
│ Land             ┆ 69020  │
│ Building         ┆ 1890   │
└──────────────────┴────────┘

property_sub_type_en:
shape: (15, 2)
┌──────────────────────┬────────┐
│ property_sub_type_en ┆ count  │
│ ---                  ┆ ---    │
│ str                  ┆ u32    │
╞══════════════════════╪════════╡
│ Flat                 ┆ 504904 │
│ null                 ┆ 89131  │
│ Villa                ┆ 61678  │
│ Office               ┆ 14424  │
│ Hotel Apartment      ┆ 11155  │
│ Hotel Rooms          ┆ 5649   │
│ Shop                 ┆ 5262   │
│ Stacked Townhouses   ┆ 122    │
│ Workshop             ┆ 45     │
│ Show Rooms           ┆ 40     │
│ Building             ┆ 9      │
│ Clinic               ┆ 8      │
│ Hotel                ┆ 6      │
│ Gymnasium       

In [15]:
null_sub_units = df.filter(
    pl.col("property_sub_type_en").is_null() & 
    (pl.col("property_type_en") == "Unit")
).height

print(f"Null sub_type + Unit rows: {null_sub_units:,}")

Null sub_type + Unit rows: 0


## Step 12: Build a Unified Property Category (Flat / Villa)

The original two-column type system is unnecessarily granular for our analysis. We collapse everything into a single `property_cat` column with two values: `Flat` and `Villa`.

The mapping logic is:
- `property_sub_type_en == "Flat"` → Flat
- Any villa/townhouse/compound sub-type → Villa
- `property_type_en == "Unit"` with a null sub-type → dropped (unclassifiable)
- Hotel apartments and "other" types → dropped (not residential sales)

This gives us a clean, binary dimension that separates the two structurally different markets throughout all downstream analysis.

In [16]:
df = df.with_columns(
    pl.when(pl.col("property_sub_type_en") == "Flat")
      .then(pl.lit("Flat"))
      .when(
          (pl.col("property_sub_type_en") == "Villa") |
          (pl.col("property_sub_type_en") == "Stacked Townhouses") |
          (pl.col("property_type_en") == "Villa")
      )
      .then(pl.lit("Villa"))
      .otherwise(None)
      .alias("property_cat")
)

df = df.filter(pl.col("property_cat").is_not_null())

print(f"Rows remaining: {df.height:,}")
print(df["property_cat"].value_counts().sort("count", descending=True))

Rows remaining: 584,933
shape: (2, 2)
┌──────────────┬────────┐
│ property_cat ┆ count  │
│ ---          ┆ ---    │
│ str          ┆ u32    │
╞══════════════╪════════╡
│ Flat         ┆ 504904 │
│ Villa        ┆ 80029  │
└──────────────┴────────┘


## Step 13: Drop the Original Type Columns

Now that `property_cat` captures the information we need, the original `property_type_en`, `property_sub_type_en`, and `property_usage_en` columns are redundant. We also drop `project_number`, which is an internal DLD identifier with no analytical value.

Keeping redundant columns creates confusion later: it is not obvious which column to use, and joins or group-bys can accidentally pull from the wrong one.

In [17]:
df = df.drop(["property_type_en", "property_sub_type_en", "property_usage_en","project_number"])

## Step 14: Inspect Area Names

We print the unique count and value counts for `area_name_en` to understand the geographic granularity of the dataset and spot any casing or formatting inconsistencies before we build the geo hierarchy.

This step reveals that the same physical area can appear under multiple slightly different names due to capitalisation inconsistencies in the source data, which motivates the normalisation in the next step.

In [18]:
print(df["area_name_en"].n_unique())
print(df["area_name_en"].value_counts().sort("count", descending=True))

133
shape: (133, 2)
┌─────────────────────────────────┬───────┐
│ area_name_en                    ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ Al Barsha South Fourth          ┆ 58500 │
│ Business Bay                    ┆ 41958 │
│ Marsa Dubai                     ┆ 35523 │
│ Wadi Al Safa 5                  ┆ 29625 │
│ Al Merkadh                      ┆ 26472 │
│ Madinat Al Mataar               ┆ 23479 │
│ Hadaeq Sheikh Mohammed Bin Ras… ┆ 23086 │
│ Burj Khalifa                    ┆ 20304 │
│ Al Thanyah Fifth                ┆ 18647 │
│ Al Khairan First                ┆ 17628 │
│ Al Barshaa South Third          ┆ 16779 │
│ Jabal Ali First                 ┆ 16623 │
│ Al Hebiah Fourth                ┆ 13701 │
│ Madinat Dubai Almelaheyah       ┆ 13127 │
│ Me'Aisem First                  ┆ 12711 │
│ Wadi Al Safa 3                  ┆ 12263 │
│ Al Barsha South Fifth           ┆ 11134 │
│ Al Yelayis

## Step 15: Build the Geographic Hierarchy (District and Neighborhood)

The DLD uses highly granular area codes. "Jumeirah Village Circle 1", "Jumeirah Village Circle 2" and so on are distinct entries but represent the same community to any buyer or analyst.

We do two things here:
1. **Normalise casing** using `str.to_titlecase()` to remove inconsistencies.
2. **Strip ordinal suffixes** using a regex that removes trailing numbers and directional qualifiers (First, Second, North, South, etc.) to consolidate micro-areas into meaningful communities. This becomes the `district` column.

`district` gives us a mid-level geographic dimension between raw area codes and the broad city level, which is the right granularity for neighborhood-level analysis.

In [19]:
# Fix casing first
df = df.with_columns(
    pl.col("area_name_en").str.to_titlecase().alias("area_name_en")
)

# Strip ordinal suffixes to consolidate sub-districts
df = df.with_columns(
    pl.col("area_name_en")
      .str.replace(
          r"\s+(First|Second|Third|Fourth|Fifth|Sixth|Seventh|\d+)$",
          "",
          literal=False
      )
      .alias("district")
)

print(f"Unique areas (original): {df['area_name_en'].n_unique()}")
print(f"Unique districts (consolidated): {df['district'].n_unique()}")

Unique areas (original): 133
Unique districts (consolidated): 85


## Step 16: Apply Brand Name Mapping (Neighborhood Column)

The DLD's official area names are often technical or archaic. "Marsa Dubai" is Dubai Marina. "Burj Khalifa" is Downtown Dubai. Analysts, investors, and the press use the market names, not the DLD codes.

We build a manual mapping dictionary that translates DLD official names to their widely-recognised brand names. This becomes the `neighborhood` column, which is what every chart in the analysis notebooks uses.

The mapping was researched and validated against the Dubai property market to ensure accuracy. Where no brand name exists, the cleaned DLD name is kept as-is.

In [20]:
BRAND_MAP = {
    # Marina & Downtown
    "Marsa Dubai":                              "Dubai Marina",
    "Burj Khalifa":                             "Downtown Dubai",

    # MBR City / Meydan
    "Al Merkadh":                               "Mohammed Bin Rashid City",
    "Hadaeq Sheikh Mohammed Bin Rashid":        "Mohammed Bin Rashid City",
    "Bukadra":                                  "Meydan",

    # Creek Harbour
    "Al Khairan First":                         "Dubai Creek Harbour",

    # Al Barsha South cluster
    "Al Barsha South Fourth":                   "Jumeirah Village Circle",
    "Al Barsha South Fifth":                    "Jumeirah Village Triangle",  # ← JVT is 5, not 3
    "Al Barshaa South Third":                   "Arjan",                      # ← not JVT

    # JLT & Emirates Hills
    "Al Thanyah Fifth":                         "Jumeirah Lake Towers",
    "Al Thanyah Third":                         "The Meadows",
    "Al Thanayah Fourth":                       "Emirates Hills",

    # Dubai South
    "Madinat Al Mataar":                        "Dubai South",

    # Dubailand / DAMAC / Arabian Ranches
    "Al Yelayiss 1":                            "DAMAC Sun City",             # ← not Dubai South
    "Al Yelayiss 2":                            "DAMAC Sun City",
    "Wadi Al Safa 2":                           "Falcon City Of Wonders",     # ← not Arabian Ranches
    "Wadi Al Safa 3":                           "Al Barari",
    "Wadi Al Safa 4":                           "Dubailand",
    "Wadi Al Safa 5":                           "Villanova",
    "Wadi Al Safa 6":                           "Arabian Ranches",            # ← AR1 is here
    "Wadi Al Safa 7":                           "Arabian Ranches 2",
    "Madinat Hind 4":                           "DAMAC Hills 2",

    # Al Hebiah cluster
    "Al Hebiah First":                          "Motor City",
    "Al Hebiah Second":                         "Dubai Sports City",
    "Al Hebiah Third":                          "DAMAC Hills",                # ← not Sports City
    "Al Hebiah Fourth":                         "Dubai Sports City",          # ← Sports City is here
    "Al Hebiah Fifth":                          "Golf City",

    # Production / Silicon / Maritime
    "Me'Aisem First":                           "Dubai Production City",
    "Nadd Hessa":                               "Dubai Silicon Oasis",
    "Madinat Dubai Almelaheyah":                "Dubai Maritime City",

    # Jebel Ali / DIP / Int'l City
    "Jabal Ali First":                          "Jebel Ali",
    "Jabal Ali":                                "Jebel Ali",
    "Dubai Investment Park Second":             "Dubai Investment Park",
    "Al Warsan First":                          "International City",

    # Spelling cleanups
    "Al Jadaf":                                 "Al Jaddaf",
    "Palm Deira":                               "Dubai Islands",
    "Zaabeel First":                            "Za'abeel",
    "Zaabeel Second":                           "Za'abeel",
    "Um Suqaim Third":                          "Umm Suqeim",
}

df = df.with_columns(
    pl.col("area_name_en")
      .replace(BRAND_MAP)
      .alias("neighborhood")
)

# Fall back to district for anything unmapped
df = df.with_columns(
    pl.when(pl.col("neighborhood").is_in(list(BRAND_MAP.values())))
      .then(pl.col("neighborhood"))
      .otherwise(pl.col("district"))
      .alias("neighborhood")
)

print(f"Unique neighborhoods: {df['neighborhood'].n_unique()}")
print(df["neighborhood"].value_counts().sort("count", descending=True))

Unique neighborhoods: 98
shape: (98, 2)
┌───────────────────────────┬───────┐
│ neighborhood              ┆ count │
│ ---                       ┆ ---   │
│ str                       ┆ u32   │
╞═══════════════════════════╪═══════╡
│ Jumeirah Village Circle   ┆ 58500 │
│ Mohammed Bin Rashid City  ┆ 49558 │
│ Business Bay              ┆ 41958 │
│ Dubai Marina              ┆ 35523 │
│ Villanova                 ┆ 29625 │
│ Dubai South               ┆ 23479 │
│ Downtown Dubai            ┆ 20304 │
│ Jebel Ali                 ┆ 19056 │
│ Jumeirah Lake Towers      ┆ 18647 │
│ Dubai Creek Harbour       ┆ 17628 │
│ Dubai Sports City         ┆ 17319 │
│ Arjan                     ┆ 16779 │
│ DAMAC Sun City            ┆ 14618 │
│ Dubai Maritime City       ┆ 13127 │
│ Dubai Production City     ┆ 12711 │
│ Al Barari                 ┆ 12263 │
│ Jumeirah Village Triangle ┆ 11134 │
│ Motor City                ┆ 9696  │
│ International City        ┆ 9658  │
│ Al Barshaa South          ┆ 9532  │
│ Palm Jum

## Step 17: Sanity Check — Sample After Geo Transformation

We sample 10 rows again to verify that the area name normalisation, district stripping, and brand mapping all applied correctly. This is a quick visual confirmation before moving on to text column cleaning.

In [21]:
df.sample(10, seed=42)

instance_date,area_name_en,building_name_en,project_name_en,master_project_en,nearest_landmark_en,nearest_metro_en,nearest_mall_en,rooms_en,has_parking,procedure_area,actual_worth,meter_sale_price,year,month,sale_type,property_cat,district,neighborhood
date,str,str,str,str,str,str,str,str,i64,f64,f64,f64,i32,i8,str,str,str,str
2025-05-20,"""Burj Khalifa""","""One Residence""","""One Residence""","""DownTown Dubai""","""Downtown Dubai""","""Buj Khalifa Dubai Mall Metro S…","""Dubai Mall""","""Studio""",1,43.68,1.3774e6,31533.88,2025,5,"""Off-Plan""","""Flat""","""Burj Khalifa""","""Downtown Dubai"""
2022-11-16,"""Al Barsha South Fourth""","""Binghatti Heights""","""Binghatti Heights""","""Jumeirah Village Circle""",null,null,null,"""1 B/R""",1,87.81,650000.0,7402.35,2022,11,"""Off-Plan""","""Flat""","""Al Barsha South""","""Jumeirah Village Circle"""
2024-02-16,"""Jabal Ali First""","""THE NOOK 2-1""","""The Nook""","""Wasl Gate""","""Expo 2020 Site""","""ENERGY Metro Station""","""Ibn-e-Battuta Mall""","""1 B/R""",1,50.26,715000.0,14226.02,2024,2,"""Ready""","""Flat""","""Jabal Ali""","""Jebel Ali"""
2023-05-25,"""Burj Khalifa""","""BD Standpoint B""","""DD STAND POINT""","""DownTown Dubai""","""Burj Khalifa""","""Buj Khalifa Dubai Mall Metro S…","""Dubai Mall""","""Studio""",1,41.52,950000.0,22880.54,2023,5,"""Ready""","""Flat""","""Burj Khalifa""","""Downtown Dubai"""
2024-10-28,"""Burj Khalifa""","""Blvd Heights T1""","""BLVD HEIGHTS""","""DownTown Dubai""","""Burj Khalifa""","""Business Bay Metro Station""","""Dubai Mall""","""2 B/R""",1,161.7,3.95e6,24427.95,2024,10,"""Ready""","""Flat""","""Burj Khalifa""","""Downtown Dubai"""
2025-08-04,"""Madinat Al Mataar""","""Azizi Venice 12 - Building B""","""azizi venice 12""","""Dubai World Central""",null,null,null,"""Studio""",1,33.49,650000.0,19408.78,2025,8,"""Off-Plan""","""Flat""","""Madinat Al Mataar""","""Dubai South"""
2024-12-18,"""Al Barsha South Fourth""","""ASTORIA RESIDENCE""","""ASTORIA RESIDNECE""","""Jumeirah Village Circle""","""Sports City Swimming Academy""","""Dubai Internet City""","""Mall of the Emirates""","""2 B/R""",1,165.2,1.3e6,7869.25,2024,12,"""Ready""","""Flat""","""Al Barsha South""","""Jumeirah Village Circle"""
2024-10-08,"""Al Barshaa South Second""","""Binghatti Hills - 2""","""Binghatti Hills ""","""Dubai Science Park""","""Motor City""","""First Abu Dhabi Bank Metro Sta…","""Mall of the Emirates""","""1 B/R""",1,82.73,1.133293e6,13698.7,2024,10,"""Off-Plan""","""Flat""","""Al Barshaa South""","""Al Barshaa South"""
2025-06-02,"""Madinat Al Mataar""","""AZIZI VENICE 10 - BUILDING B""","""AZIZI VENICE 10""","""Dubai World Central""",null,null,null,"""1 B/R""",1,64.01,959000.0,14982.03,2025,6,"""Off-Plan""","""Flat""","""Madinat Al Mataar""","""Dubai South"""


## Step 18: Normalise Building and Project Name Text

`building_name_en` and `project_name_en` arrive with inconsistent casing (all-caps, all-lowercase, mixed) and leading/trailing whitespace. We strip and apply title case to both.

This is purely cosmetic but important for display quality in any chart that surfaces these values, and it also prevents the same building appearing as two distinct entries in a group-by due to case differences.

In [22]:
df = df.with_columns([
    pl.col("building_name_en")
      .str.strip_chars()
      .str.to_titlecase()
      .alias("building_name_en"),

    pl.col("project_name_en")
      .str.strip_chars()
      .str.to_titlecase()
      .alias("project_name_en"),
])

# Sanity check
print(df.select(["building_name_en", "project_name_en"]).sample(10, seed=42))

shape: (10, 2)
┌──────────────────────────────┬───────────────────┐
│ building_name_en             ┆ project_name_en   │
│ ---                          ┆ ---               │
│ str                          ┆ str               │
╞══════════════════════════════╪═══════════════════╡
│ One Residence                ┆ One Residence     │
│ Binghatti Heights            ┆ Binghatti Heights │
│ The Nook 2-1                 ┆ The Nook          │
│ Bd Standpoint B              ┆ Dd Stand Point    │
│ Blvd Heights T1              ┆ Blvd Heights      │
│ Azizi Venice 12 - Building B ┆ Azizi Venice 12   │
│ Astoria Residence            ┆ Astoria Residnece │
│ Binghatti Hills - 2          ┆ Binghatti Hills   │
│ Azizi Venice 10 - Building B ┆ Azizi Venice 10   │
│ Binghatti Luna               ┆ Binghatti Luna    │
└──────────────────────────────┴───────────────────┘


## Step 19: Inspect and Map Bedroom Counts

`rooms_en` is a free-text field containing values like "Studio", "1 B/R", "2 B/R", "Shop", "Office", and "Penthouse". We need to convert this into a clean integer `bedrooms` column.

First we print value counts to see the full range of values and their frequency. This tells us which values are legitimate bedroom counts, which are commercial slipthrough (Shop, Office), and which are edge cases needing a decision (Penthouse).

In [23]:
print(df["rooms_en"].value_counts().sort("count", descending=True))
print(f"\nNulls: {df['rooms_en'].is_null().sum():,} ({df['rooms_en'].is_null().mean()*100:.1f}%)")

shape: (14, 2)
┌─────────────┬────────┐
│ rooms_en    ┆ count  │
│ ---         ┆ ---    │
│ str         ┆ u32    │
╞═════════════╪════════╡
│ 1 B/R       ┆ 220635 │
│ 2 B/R       ┆ 136632 │
│ Studio      ┆ 109261 │
│ 3 B/R       ┆ 66617  │
│ 4 B/R       ┆ 28993  │
│ null        ┆ 18455  │
│ 5 B/R       ┆ 3745   │
│ 6 B/R       ┆ 249    │
│ PENTHOUSE   ┆ 233    │
│ Single Room ┆ 57     │
│ 7 B/R       ┆ 43     │
│ Shop        ┆ 8      │
│ 9 B/R       ┆ 3      │
│ Office      ┆ 2      │
└─────────────┴────────┘

Nulls: 18,455 (3.2%)


## Step 20: Clean and Convert Bedroom Column

Based on the inspection above, we apply three cleaning decisions:

1. **Drop commercial rows** — "Shop" and "Office" entries slipped through the property type filter. These are not residential transactions and are removed.
2. **Flag Penthouses** — Penthouses are legitimate residential units but their bedroom count is not explicit in `rooms_en`. We flag them temporarily, then drop them rather than guess a bedroom count.
3. **Map text to integers** — "Studio" becomes `0`, "1 B/R" becomes `1`, and so on up through "9 B/R". Any remaining nulls after mapping are dropped.

The result is a clean `bedrooms` column as an integer, ready for numeric analysis and sorting.

In [24]:
# Drop commercial slipthrough rows
df = df.filter(~pl.col("rooms_en").is_in(["Shop", "Office"]))

# Flag penthouses before converting
df = df.with_columns(
    (pl.col("rooms_en") == "PENTHOUSE").alias("is_penthouse")
)

# Map to integer bedrooms
bedroom_map = {
    "Studio":      0,
    "Single Room": 0,
    "1 B/R":       1,
    "2 B/R":       2,
    "3 B/R":       3,
    "4 B/R":       4,
    "5 B/R":       5,
    "6 B/R":       6,
    "7 B/R":       7,
    "9 B/R":       9,
    "PENTHOUSE":   None,
}

df = df.with_columns(
    pl.col("rooms_en").replace_strict(bedroom_map, default=None)
      .cast(pl.Int8)
      .alias("bedrooms")
).drop("rooms_en")

print(df["bedrooms"].value_counts().sort("bedrooms"))
print(f"\nPenthouses: {df['is_penthouse'].sum():,}")

shape: (10, 2)
┌──────────┬────────┐
│ bedrooms ┆ count  │
│ ---      ┆ ---    │
│ i8       ┆ u32    │
╞══════════╪════════╡
│ null     ┆ 233    │
│ 0        ┆ 109318 │
│ 1        ┆ 220635 │
│ 2        ┆ 136632 │
│ 3        ┆ 66617  │
│ 4        ┆ 28993  │
│ 5        ┆ 3745   │
│ 6        ┆ 249    │
│ 7        ┆ 43     │
│ 9        ┆ 3      │
└──────────┴────────┘

Penthouses: 233


In [25]:
# Drop penthouse

df = df.filter(pl.col("bedrooms").is_not_null())
df = df.drop("is_penthouse")

## Step 21: Final Sanity Check

One last sample before saving. After 20 transformation steps, we verify that the final dataframe looks exactly as expected: correct column names, correct types, no stray nulls in critical fields, and sensible values throughout.

In [26]:
# Sanity check
df.sample(10, seed=42)

instance_date,area_name_en,building_name_en,project_name_en,master_project_en,nearest_landmark_en,nearest_metro_en,nearest_mall_en,has_parking,procedure_area,actual_worth,meter_sale_price,year,month,sale_type,property_cat,district,neighborhood,bedrooms
date,str,str,str,str,str,str,str,i64,f64,f64,f64,i32,i8,str,str,str,str,i8
2023-11-16,"""Burj Khalifa""","""Bellevue Towers-1""","""Bellevue Towers""","""DownTown Dubai""","""Downtown Dubai""","""Business Bay Metro Station""","""Dubai Mall""",1,120.41,2.55e6,21177.64,2023,11,"""Off-Plan""","""Flat""","""Burj Khalifa""","""Downtown Dubai""",2
2025-02-25,"""Al Yufrah 1""",null,"""The Valley - Elea""","""The Valley""",null,null,null,0,288.19,3.546888e6,12307.46,2025,2,"""Off-Plan""","""Villa""","""Al Yufrah""","""Al Yufrah""",4
2021-10-15,"""Business Bay""","""Urban Oasis""","""Urban Oasis""","""Business Bay""","""Downtown Dubai""","""Business Bay Metro Station""","""Dubai Mall""",1,72.83,1.367446e6,18775.86,2021,10,"""Off-Plan""","""Flat""","""Business Bay""","""Business Bay""",1
2025-10-08,"""Jabal Ali First""","""The Horizon At Sobha Central""","""Sobha Central Phase I""","""DMCC-EZ2""","""Sports City Swimming Academy""","""Harbour Tower""","""Marina Mall""",1,58.68,1.75909e6,29977.68,2025,10,"""Off-Plan""","""Flat""","""Jabal Ali""","""Jebel Ali""",1
2024-11-28,"""Al Thanyah Fifth""","""Concorde Tower 1""","""Concorde Tower""","""Jumeirah Lakes Towers""","""Sports City Swimming Academy""","""Jumeirah Lakes Towers""","""Marina Mall""",1,65.46,1e6,15276.5,2024,11,"""Ready""","""Flat""","""Al Thanyah""","""Jumeirah Lake Towers""",1
2023-11-01,"""Al Barshaa South Third""","""Divine Living""","""Divine Living""","""Arjan""","""Motor City""","""Sharaf Dg Metro Station""","""Mall of the Emirates""",1,75.77,782790.0,10331.13,2023,11,"""Off-Plan""","""Flat""","""Al Barshaa South""","""Arjan""",1
2023-02-03,"""Al Barsha South Fourth""","""Binghatti Crest""","""Binghatti Crest""","""Jumeirah Village Circle""","""Sports City Swimming Academy""","""Nakheel Metro Station""","""Marina Mall""",1,113.42,833000.0,7344.38,2023,2,"""Off-Plan""","""Flat""","""Al Barsha South""","""Jumeirah Village Circle""",3
2026-01-27,"""Al Barshaa South Third""","""Torino By Oro24 - 4""","""Torino By Oro24""","""Arjan""","""Motor City""","""Sharaf Dg Metro Station""","""Mall of the Emirates""",1,77.07,1.36e6,17646.3,2026,1,"""Off-Plan""","""Flat""","""Al Barshaa South""","""Arjan""",2
2025-10-06,"""Madinat Dubai Almelaheyah""","""Baystar By Vida Tower 1""","""Baystar By Vida""","""Mina Rashid""",null,null,null,1,67.65,2.117499e6,31300.8,2025,10,"""Off-Plan""","""Flat""","""Madinat Dubai Almelaheyah""","""Dubai Maritime City""",1


## Step 22: Save the Clean Snapshot to Parquet

We write the cleaned dataframe to a Parquet file rather than CSV for two reasons:

1. **Type preservation** — Parquet stores the schema alongside the data. When we reload in `02_analysis.ipynb` or `03_insights.ipynb`, every column comes back with the correct type already set. No re-casting, no re-parsing dates.
2. **Speed** — Parquet is a columnar binary format. Reloading 566K rows from Parquet takes a fraction of the time it takes to re-parse the 1.6M row CSV and re-run 20 cleaning steps.

This snapshot is the single source of truth for all downstream notebooks.

In [27]:
SNAPSHOT_PATH = r"E:\Databard\DLD\dld-clean.parquet"

df.write_parquet(SNAPSHOT_PATH)

print(f"Saved to {SNAPSHOT_PATH}")
print(f"Rows: {df.height:,} | Columns: {df.width}")
print(df.columns)


#From next session onwards, load with:


#df = pl.read_parquet(r"E:\Databard\DLD\dld-clean.parquet")

Saved to E:\Databard\DLD\dld-clean.parquet
Rows: 566,235 | Columns: 19
['instance_date', 'area_name_en', 'building_name_en', 'project_name_en', 'master_project_en', 'nearest_landmark_en', 'nearest_metro_en', 'nearest_mall_en', 'has_parking', 'procedure_area', 'actual_worth', 'meter_sale_price', 'year', 'month', 'sale_type', 'property_cat', 'district', 'neighborhood', 'bedrooms']
